# Continue fine-tuning Grounding DINO: more DIOR-RSVG epochs + VRSBench referring data

**This is v2 of `kaggle_finetune_grounding_dino_dior_rsvg.ipynb`** (that notebook trained the
`dior_rsvg_finetuned.pth` checkpoint currently in `models/grounding/checkpoints/`). This one:

1. **Resumes from that checkpoint** (via Open-GroundingDino's `PRETRAIN_MODEL_PATH`, which loads
   weights and starts a fresh training schedule -- not `--resume`, which would also restore
   optimizer/epoch state from an interrupted run; that's not what we want here) instead of
   starting over from the raw Swin-T pretrained weights.
2. **Adds VRSBench's `[refer]` training examples** (36,313 referring expressions, verified by
   parsing `VRSBench_train.json` locally before writing this notebook) as a second training source
   alongside DIOR-RSVG -- Open-GroundingDino's ODVG dataset config natively supports a *list* of
   `{root, anno}` sources for `train`, so the two datasets are trained on together without needing
   to physically merge them into one file or share an image directory.
3. **Evaluates three ways** at the end -- zero-shot base Swin-T, your *current* checkpoint (the
   one this run started from), and the *new* checkpoint -- so you can actually see whether the
   extra training helped before overwriting anything locally.

**You need to upload your current checkpoint as a Kaggle input before running this** -- see the
cell right after the pretrained-weights download below for exact steps.

Run cells top to bottom. Kaggle: enable **Internet** and a **GPU** (T4x2 or P100) in the notebook's
Settings panel before starting -- same as the original notebook.

In [ ]:
import torch, subprocess
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" -", torch.cuda.get_device_name(i))


## 1. Setup — clone Open-GroundingDino, install deps, build the CUDA ops

If the CUDA-op build fails, it's almost always a CUDA/PyTorch version mismatch. Common fix: check
`nvcc --version` vs `torch.version.cuda` and, if they disagree, either install a matching `nvcc`
via `conda install -c nvidia cuda-nvcc=<version>` or pin `torch` to match the system CUDA before
re-running the build.

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/longzw1997/Open-GroundingDino.git
%cd Open-GroundingDino
!pip install -q -r requirements.txt
# Open-GroundingDino calls BertModel.get_head_mask, which transformers 5.x removed -- the
# Kaggle image now ships 5.x, so training dies building the model without this pin.
!pip install -q "transformers<5"
!pip install -q gdown pycocotools


In [ ]:
%cd /kaggle/working/Open-GroundingDino/models/GroundingDINO/ops
!python setup.py build install
!python test.py   # should print a bunch of "True" — confirms the compiled op matches the pure-pytorch fallback
%cd /kaggle/working/Open-GroundingDino


## 2. Download pretrained weights (Swin-T checkpoint + warm the BERT cache)

In [ ]:
%cd /kaggle/working/Open-GroundingDino
!mkdir -p weights
!wget -q -P weights https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
print("Downloaded:", __import__("os").path.getsize("weights/groundingdino_swint_ogc.pth"), "bytes")

# bert-base-uncased auto-downloads from the HF hub the first time the model is built (public, no
# token needed) -- warm the cache here so training doesn't stall on it mid-run.
from transformers import AutoTokenizer, AutoModel
AutoTokenizer.from_pretrained("bert-base-uncased")
AutoModel.from_pretrained("bert-base-uncased")
print("BERT cached.")


## 2b. Upload your CURRENT fine-tuned checkpoint

Before running this cell: on the right-hand panel of the Kaggle notebook editor, click
**Add Input -> Upload -> New Dataset**, upload your local
`models/grounding/checkpoints/dior_rsvg_finetuned.pth`, and name the dataset something like
`dior-rsvg-finetuned-v1`. It'll then be mounted read-only under
`/kaggle/input/dior-rsvg-finetuned-v1/dior_rsvg_finetuned.pth` (Kaggle lowercases/hyphenates the
dataset name for the path -- check the actual path in the right-hand panel after upload and adjust
`CURRENT_CKPT_GLOB` below if it doesn't match).


In [ ]:
import glob

CURRENT_CKPT_GLOB = "/kaggle/input/*/dior_rsvg_finetuned.pth"
matches = glob.glob(CURRENT_CKPT_GLOB)
assert matches, (
    f"No checkpoint found matching {CURRENT_CKPT_GLOB} -- did you upload it as a Kaggle input "
    "dataset yet? See the markdown cell above."
)
CURRENT_CKPT = matches[0]
print("Using current checkpoint:", CURRENT_CKPT)


## 3. Download DIOR-RSVG

Canonical source: the DIOR-RSVG authors' Google Drive folder (linked from
https://github.com/ZhanYang-nwpu/RSVG-pytorch). `gdown --folder` pulls the whole folder; if it
stalls or hits Google's "too many downloads" warning, re-run the cell (gdown resumes) or grab the
zip manually and upload it as a Kaggle Dataset instead — either way you want the layout below.

Expected layout:
```
DIOR_RSVG/
  Annotations/   *.xml   (bbox + referring expression per object)
  JPEGImages/    *.jpg
  train.txt      (26991 object-level indices)
  val.txt        (3829)
  test.txt       (7500)
```

In [ ]:
%cd /kaggle/working
!gdown --folder "https://drive.google.com/drive/folders/1hTqtYsC6B-m4ED2ewx5oKuYZV13EoJp_" -O DIOR_RSVG
!echo "---"
!find DIOR_RSVG -maxdepth 2 | head -20

# gdown pulls the folder's contents as-is -- sometimes that's already-extracted files, sometimes
# it's zip archives (Annotations.zip / JPEGImages.zip) that still need unzipping. Either way, any
# zip found gets extracted then DELETED immediately -- keeping both the zip and its extracted
# contents on disk at once was blowing past Kaggle's working-directory quota on a live run.
import glob, os
for zip_path in glob.glob("DIOR_RSVG/*.zip"):
    print(f"Extracting and removing {zip_path} ...")
    !unzip -q -o {zip_path} -d DIOR_RSVG
    os.remove(zip_path)
!echo "--- after extraction ---"
!find DIOR_RSVG -maxdepth 2 | head -20
!du -sh DIOR_RSVG


## 4. Parse the XML annotations into ODVG grounding JSONL

Mirrors the official `data_loader.py` exactly: for each `<object>` in each XML file,
`member[0]`=category name, `member[2]`=`bndbox` (xmin,ymin,xmax,ymax), `member[3]`=the referring
expression. `train.txt` / `val.txt` / `test.txt` are indices into this flattened
(image, object)-pair list, walked in the same sorted-filename order the original loader uses --
**verified** against the actual source
(github.com/ZhanYang-nwpu/RSVG-pytorch/blob/main/data_loader.py, fetched and compared directly):
same `os.walk` + full-path sort over the Annotations XMLs, same per-file `root.findall('object')`
order, same flat monotonic `count` used as the index the split `.txt` files reference. The only
difference is deliberate and behavior-preserving -- parsing all objects first and filtering by
`index in {train,val,test}_ids` (a set) afterward, instead of the original's single-pass
`if count in Index` (a list) -- identical resulting split membership either way.

One JSONL line per referring expression (not grouped by image) — this matches how DIOR-RSVG,
RSVG-HR, and OPT-RSVG are all trained/evaluated in the literature: one (image, query, box) triplet
per sample.

In [ ]:
import os, json, pickle
import xml.etree.ElementTree as ET
from PIL import Image

DIOR_ROOT = "/kaggle/working/DIOR_RSVG"
ANNO_DIR  = os.path.join(DIOR_ROOT, "Annotations")
IMG_DIR   = os.path.join(DIOR_ROOT, "JPEGImages")

def load_split_ids(split):
    with open(os.path.join(DIOR_ROOT, f"{split}.txt")) as f:
        return set(int(x.strip()) for x in f if x.strip())

def get_image_size(xml_root, image_path):
    w_el, h_el = xml_root.find("./size/width"), xml_root.find("./size/height")
    if w_el is not None and h_el is not None:
        return int(w_el.text), int(h_el.text)
    with Image.open(image_path) as im:
        return im.size  # (width, height)

def parse_all_objects():
    xml_files = sorted(
        os.path.join(dp, f) for dp, _, fs in os.walk(ANNO_DIR) for f in fs if f.endswith(".xml")
    )
    records, count = [], 0
    for xp in xml_files:
        root = ET.parse(xp).getroot()
        filename = root.find("./filename").text
        w, h = get_image_size(root, os.path.join(IMG_DIR, filename))
        for member in root.findall("object"):
            category = member[0].text
            x1, y1, x2, y2 = (float(member[2][0].text), float(member[2][1].text),
                               float(member[2][2].text), float(member[2][3].text))
            expression = member[3].text
            records.append(dict(index=count, filename=filename, category=category,
                                 bbox=[x1, y1, x2, y2], width=w, height=h, expression=expression))
            count += 1
    return records

records = parse_all_objects()
n_images = len({r["filename"] for r in records})
print(f"Parsed {len(records)} (image, expression) pairs across {n_images} images")
categories = sorted({r["category"] for r in records})
print(f"{len(categories)} categories:", categories)


In [ ]:
def to_odvg_line(rec):
    x1, y1, x2, y2 = rec["bbox"]
    return json.dumps({
        "filename": rec["filename"],
        "height": rec["height"],
        "width": rec["width"],
        "grounding": {
            "caption": rec["expression"],
            "regions": [{"bbox": [x1, y1, x2, y2], "phrase": rec["expression"]}],
        },
    })

train_ids, val_ids, test_ids = load_split_ids("train"), load_split_ids("val"), load_split_ids("test")

train_lines = [to_odvg_line(r) for r in records if r["index"] in train_ids]
val_lines   = [to_odvg_line(r) for r in records if r["index"] in val_ids]
test_records = [r for r in records if r["index"] in test_ids]

os.makedirs("/kaggle/working/data", exist_ok=True)
with open("/kaggle/working/data/dior_rsvg_train_grounding.jsonl", "w") as f:
    f.write("\n".join(train_lines))
with open("/kaggle/working/data/dior_rsvg_val_grounding.jsonl", "w") as f:
    f.write("\n".join(val_lines))
with open("/kaggle/working/data/dior_rsvg_test_records.pkl", "wb") as f:
    pickle.dump(test_records, f)

print(f"train={len(train_lines)} (paper: 26991)  val={len(val_lines)} (paper: 3829)  "
      f"test={len(test_records)} (paper: 7500)")


## 4b. Download VRSBench and convert its `[refer]` examples to the same ODVG format

VRSBench (https://huggingface.co/datasets/xiang709/VRSBench, CC-BY-4.0) ships train annotations as
one LLaVA-style conversation file (`VRSBench_train.json`) with `[caption]` / `[refer]` / `[vqa]`
tagged turns -- we only want the 36,313 `[refer]` ones here. Verified against the actual file
before writing this cell:

- Human turn: `[refer] ... <p>{referring expression}</p> ...`
- GPT turn: `{<x1><y1><x2><y2>}`, coordinates **normalized 0-100** per the dataset authors' README
  (https://github.com/lx709/VRSBench) -- *not* 0-1 or absolute pixels.
- ~5.9% of boxes have a coordinate outside [0, 100] (annotation noise, includes some negative
  values) -- clip to [0, 100] rather than discard, then drop any box that's degenerate after
  clipping (x2<=x1 or y2<=y1).

Images come from `Images_train.zip` (8.4GB) -- downloaded here, inside Kaggle, not locally.


In [ ]:
%cd /kaggle/working
!wget -q https://huggingface.co/datasets/xiang709/VRSBench/resolve/main/VRSBench_train.json

# Kaggle gives 20GB of working-directory space. Images_train.zip is 8.4GB and the full extraction
# is another ~8GB -- both have to coexist during unzip, which blows the quota before any cleanup
# can run (that's what killed this cell before). The [refer] annotations touch 99.8% of the 20,262
# training images, so we cap the number of IMAGES instead and extract only those.
#
# MAX_IMAGES=4000 -> ~1.6GB extracted, ~10GB peak during extraction, ~1.6GB after the zip is
# deleted. Raise it if you have headroom (each image averages 0.40MB); lower it if training
# checkpoints later crowd the disk.
import json, os, random, zipfile

MAX_IMAGES = 4000

with open("VRSBench_train.json") as f:
    vrsbench_all = json.load(f)

refer_all = [d for d in vrsbench_all if "[refer]" in d["conversations"][0]["value"]]
all_refer_images = sorted({d["image"] for d in refer_all})
random.Random(0).shuffle(all_refer_images)
SUBSET_IMAGES = set(all_refer_images[:MAX_IMAGES])
print(f"{len(refer_all)} [refer] annotations across {len(all_refer_images)} images; "
      f"keeping {len(SUBSET_IMAGES)} images")

!wget -q https://huggingface.co/datasets/xiang709/VRSBench/resolve/main/Images_train.zip
!df -h /kaggle/working | tail -1

VRSBENCH_IMG_DIR = "/kaggle/working/VRSBench_Images_train"
os.makedirs(VRSBENCH_IMG_DIR, exist_ok=True)

extracted = 0
with zipfile.ZipFile("Images_train.zip") as z:
    for info in z.infolist():
        basename = os.path.basename(info.filename)
        if basename in SUBSET_IMAGES:
            info.filename = basename  # flatten -- drop the "Images_train/" prefix
            z.extract(info, VRSBENCH_IMG_DIR)
            extracted += 1
print(f"Extracted {extracted} images")

os.remove("Images_train.zip")  # reclaim 8.4GB immediately -- do NOT leave this until later
!du -sh {VRSBENCH_IMG_DIR}
!df -h /kaggle/working | tail -1


In [ ]:
import json, re
from PIL import Image

p_pattern = re.compile(r"<p>(.*?)</p>")
box_pattern = re.compile(r"\{<(-?\d+)><(-?\d+)><(-?\d+)><(-?\d+)>\}")

# Only the annotations whose image actually got extracted above.
refer_items = [d for d in refer_all if d["image"] in SUBSET_IMAGES]
print(f"{len(refer_items)} [refer] annotations for the {len(SUBSET_IMAGES)} extracted images")

vrsbench_lines = []
skipped_no_image = 0
skipped_degenerate = 0
image_size_cache = {}

for item in refer_items:
    human = item["conversations"][0]["value"]
    gpt = item["conversations"][1]["value"]
    p_match = p_pattern.search(human)
    box_match = box_pattern.search(gpt)
    if not p_match or not box_match:
        continue

    filename = item["image"]
    if filename not in image_size_cache:
        try:
            with Image.open(f"{VRSBENCH_IMG_DIR}/{filename}") as im:
                image_size_cache[filename] = im.size  # (width, height)
        except FileNotFoundError:
            skipped_no_image += 1
            continue
    w, h = image_size_cache[filename]

    # Coordinates are normalized 0-100 per the VRSBench authors' README; ~5.9% fall outside that
    # range (including negatives), so clip rather than discard, then drop anything degenerate.
    x1, y1, x2, y2 = (max(0, min(100, int(v))) for v in box_match.groups())
    if x2 <= x1 or y2 <= y1:
        skipped_degenerate += 1
        continue
    x1, y1, x2, y2 = x1 / 100 * w, y1 / 100 * h, x2 / 100 * w, y2 / 100 * h

    phrase = p_match.group(1)
    vrsbench_lines.append(json.dumps({
        "filename": filename,
        "height": h,
        "width": w,
        "grounding": {"caption": phrase, "regions": [{"bbox": [x1, y1, x2, y2], "phrase": phrase}]},
    }))

print(f"Converted {len(vrsbench_lines)} usable examples "
      f"(skipped {skipped_no_image} missing images, {skipped_degenerate} degenerate boxes)")

with open("/kaggle/working/data/vrsbench_train_grounding.jsonl", "w") as f:
    f.write("\n".join(vrsbench_lines))


## 5. Build a small COCO-format val set (for Open-GroundingDino's built-in periodic eval)

Their training loop's periodic validation only supports COCO-format detection data (fixed
categories, not free-text queries) — see the README. This is **only a training-time sanity signal**
("is box quality trending the right way"), not the metric that actually matters for your use case.
The real grounding accuracy (Acc@0.5 / Acc@0.7 / mIoU on text queries) is computed separately in
Section 9, after training, on the untouched test split.

Capped to a few hundred images to keep the periodic eval fast during a "quick" run — bump
`MAX_VAL_IMAGES` up if you want a more thorough in-training signal.

In [ ]:
MAX_VAL_IMAGES = 500

val_records = [r for r in records if r["index"] in val_ids]
cat2id = {c: i for i, c in enumerate(categories)}

val_by_image = {}
for r in val_records:
    val_by_image.setdefault(r["filename"], []).append(r)
val_image_names = list(val_by_image.keys())[:MAX_VAL_IMAGES]

images, annotations, ann_id = [], [], 0
for img_id, fname in enumerate(val_image_names):
    objs = val_by_image[fname]
    images.append({"id": img_id, "file_name": fname, "height": objs[0]["height"], "width": objs[0]["width"]})
    for r in objs:
        x1, y1, x2, y2 = r["bbox"]
        annotations.append({
            "id": ann_id, "image_id": img_id, "category_id": cat2id[r["category"]],
            "bbox": [x1, y1, x2 - x1, y2 - y1], "area": (x2 - x1) * (y2 - y1), "iscrowd": 0,
        })
        ann_id += 1

coco_val = {
    "images": images,
    "annotations": annotations,
    "categories": [{"id": i, "name": c} for c, i in cat2id.items()],
}
with open("/kaggle/working/data/dior_rsvg_val_coco.json", "w") as f:
    json.dump(coco_val, f)
print(f"COCO val: {len(images)} images, {len(annotations)} boxes, {len(categories)} categories")


## 6. Point Open-GroundingDino at the data

`train` is a *list* -- Open-GroundingDino's ODVG loader natively supports multiple `{root, anno}`
sources trained on together, so DIOR-RSVG and VRSBench are listed side by side here rather than
merged into one file (they don't share an image root anyway).


In [ ]:
dataset_cfg = {
    "train": [
        {
            "root": "/kaggle/working/DIOR_RSVG/JPEGImages/",
            "anno": "/kaggle/working/data/dior_rsvg_train_grounding.jsonl",
            "dataset_mode": "odvg",
        },
        {
            "root": VRSBENCH_IMG_DIR + "/",
            "anno": "/kaggle/working/data/vrsbench_train_grounding.jsonl",
            "dataset_mode": "odvg",
        },
    ],
    "val": [{
        "root": "/kaggle/working/DIOR_RSVG/JPEGImages/",
        "anno": "/kaggle/working/data/dior_rsvg_val_coco.json",
        "label_map": None,
        "dataset_mode": "coco",
    }],
}
with open("/kaggle/working/Open-GroundingDino/config/datasets_dior_rsvg_v2.json", "w") as f:
    json.dump(dataset_cfg, f, indent=2)
print("Wrote config/datasets_dior_rsvg_v2.json")
print(f"Training on {len(train_lines)} DIOR-RSVG + {len(vrsbench_lines)} VRSBench referring examples")


## 7. Patch `config/cfg_odvg.py`

Per the README, evaluating on a non-COCO custom set needs `use_coco_eval = False` plus a
`label_list` of your class names. Both are applied programmatically below (the second line is just
*appended* — these config files are executed top-to-bottom as plain Python, so a later assignment
safely overrides anything set earlier).

**Do this part by hand once, right after running the cell below:** open
`Open-GroundingDino/config/cfg_odvg.py` and sanity-check (adjust for a T4/P100 + "quick run"):
- `batch_size` — start small (e.g. 2-4 per GPU) and raise it only if you don't hit an OOM.
- `epochs` — 5-10 is a reasonable first pass; the published fine-tuning writeups on this model see
  best validation performance in that range, with over-detection creeping in past it.
- `lr` / `lr_backbone` — the repo's defaults are a reasonable starting point; only touch these if
  loss is clearly diverging or stuck.

In [ ]:
cfg_path = "/kaggle/working/Open-GroundingDino/config/cfg_odvg.py"
with open(cfg_path) as f:
    cfg_text = f.read()

if "use_coco_eval = True" in cfg_text:
    cfg_text = cfg_text.replace("use_coco_eval = True", "use_coco_eval = False")
    with open(cfg_path, "w") as f:
        f.write(cfg_text)
    print("Set use_coco_eval = False")
else:
    print("WARNING: 'use_coco_eval = True' not found verbatim in cfg_odvg.py — "
          "open the file and set use_coco_eval = False by hand.")

with open(cfg_path, "a") as f:
    f.write(f"\nlabel_list = {categories!r}\n")
print("Appended label_list with", len(categories), "categories")

print("\n---- now open", cfg_path, "and check batch_size / epochs / lr as noted above ----")

# Continued fine-tuning on a constrained Kaggle session. The repo defaults (epochs=15, lr=1e-4,
# save_checkpoint_interval=1) would retrain for hours at a learning rate high enough to wash out
# the DIOR-RSVG adaptation, and write a ~2-3GB checkpoint every epoch into ~8GB of free disk.
with open(cfg_path, "a") as f:
    f.write("""
epochs = 3
lr = 2e-5
lr_backbone = 2e-6
lr_drop = 2
batch_size = 2
save_checkpoint_interval = 3
""")
print("Applied continued-fine-tuning overrides (epochs=3, lr=2e-5, batch_size=2, save every 3)")


## 8. Train

Same as the original notebook, except `PRETRAIN_MODEL_PATH` now points at your *current* fine-tuned
checkpoint (uploaded in step 2b) instead of the raw Swin-T weights -- Open-GroundingDino's
`train_dist.sh` reads this from the environment (see the `PRETRAIN_MODEL_PATH=${PRETRAIN_MODEL_PATH:-...}`
line in the repo's `train_dist.sh`, confirmed by reading the actual script before writing this cell).
Consider fewer epochs than a from-scratch run (e.g. 3-5) -- you're refining an already fine-tuned
model, not starting cold.


In [ ]:
%cd /kaggle/working/Open-GroundingDino
import torch
GPU_NUM = max(1, torch.cuda.device_count())
print("Training with", GPU_NUM, "GPU(s)")


In [ ]:
import os
%cd /kaggle/working/Open-GroundingDino
os.environ["PRETRAIN_MODEL_PATH"] = CURRENT_CKPT
print("PRETRAIN_MODEL_PATH =", os.environ["PRETRAIN_MODEL_PATH"])
!bash train_dist.sh {GPU_NUM} config/cfg_odvg.py config/datasets_dior_rsvg_v2.json ./logs/dior_rsvg_run2


In [ ]:
# Confirm the actual checkpoint filename(s) written -- don't assume it, just look:
!ls -la /kaggle/working/Open-GroundingDino/logs/dior_rsvg_run2


## 9. Grounding accuracy: zero-shot vs. your current checkpoint vs. the new one

Same Acc@0.5 / Acc@0.7 / mIoU protocol as the original notebook's Section 9, evaluated on the same
held-out DIOR-RSVG test split -- but now three ways, so you can see whether this run actually
improved on what you already had before overwriting `dior_rsvg_finetuned.pth` locally.


In [ ]:
import sys, os, pickle
sys.path.insert(0, "/kaggle/working/Open-GroundingDino")
import torch
from groundingdino.util.inference import load_model, load_image, predict

def box_cxcywh_to_xyxy_abs(box_norm, w, h):
    cx, cy, bw, bh = box_norm
    return [(cx - bw/2) * w, (cy - bh/2) * h, (cx + bw/2) * w, (cy + bh/2) * h]

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def evaluate(ckpt_path, cfg_path, test_records, image_root, box_th=0.25, text_th=0.25, limit=None):
    model = load_model(cfg_path, ckpt_path)
    ious = []
    subset = test_records[:limit] if limit else test_records
    for i, r in enumerate(subset):
        img_path = os.path.join(image_root, r["filename"])
        image_source, image = load_image(img_path)
        boxes, logits, phrases = predict(model=model, image=image, caption=r["expression"],
                                          box_threshold=box_th, text_threshold=text_th)
        if len(boxes) == 0:
            ious.append(0.0)
            continue
        best_idx = int(logits.argmax())
        pred_xyxy = box_cxcywh_to_xyxy_abs(boxes[best_idx].tolist(), r["width"], r["height"])
        ious.append(iou_xyxy(pred_xyxy, r["bbox"]))
        if i % 500 == 0:
            print(i, "/", len(subset))
    n = len(ious)
    acc5 = sum(x >= 0.5 for x in ious) / n
    acc7 = sum(x >= 0.7 for x in ious) / n
    miou = sum(ious) / n
    return {"n": n, "Acc@0.5": acc5, "Acc@0.7": acc7, "mIoU": miou}

with open("/kaggle/working/data/dior_rsvg_test_records.pkl", "rb") as f:
    test_records = pickle.load(f)

image_root = "/kaggle/working/DIOR_RSVG/JPEGImages"
cfg_path = "/kaggle/working/Open-GroundingDino/tools/GroundingDINO_SwinT_OGC.py"

print("Zero-shot baseline (pretrained, not fine-tuned) on a 1000-item subset:")
print(evaluate("/kaggle/working/Open-GroundingDino/weights/groundingdino_swint_ogc.pth",
                cfg_path, test_records, image_root, limit=1000))


In [ ]:
print("Your CURRENT checkpoint (before this run) on the same 1000-item subset:")
print(evaluate(CURRENT_CKPT, cfg_path, test_records, image_root, limit=1000))


In [ ]:
# Fill in the actual checkpoint filename from the `ls` output in Section 8 above.
NEW_CKPT = "/kaggle/working/Open-GroundingDino/logs/dior_rsvg_run2/checkpoint_best_regular.pth"

print("NEW checkpoint (this run) on the same 1000-item subset:")
print(evaluate(NEW_CKPT, cfg_path, test_records, image_root, limit=1000))


## 10. Export

Exported as `dior_rsvg_finetuned_v2.pth` (not overwriting the `v1` name) so you can compare both
locally before deciding whether to actually replace
`models/grounding/checkpoints/dior_rsvg_finetuned.pth` -- only do that once the eval numbers above
actually look better than your current checkpoint's.


In [ ]:
import shutil, os
os.makedirs("/kaggle/working/output_model", exist_ok=True)
shutil.copy(NEW_CKPT, "/kaggle/working/output_model/dior_rsvg_finetuned_v2.pth")
shutil.copy("/kaggle/working/Open-GroundingDino/tools/GroundingDINO_SwinT_OGC.py",
            "/kaggle/working/output_model/GroundingDINO_SwinT_OGC.py")
print("Exported to /kaggle/working/output_model/ -- download from this notebook's Output tab.")


## Next

Download `dior_rsvg_finetuned_v2.pth` from this notebook's Output tab. Compare the three eval
numbers from Section 9 above; if `v2` beats your current checkpoint, replace
`models/grounding/checkpoints/dior_rsvg_finetuned.pth` with it locally (same `GroundingDINO_SwinT_OGC.py`
config either way -- the architecture didn't change). If not, keep your current checkpoint and
consider more epochs or a lower learning rate next time.
